## Setup

In [1]:
from gut_former.utils.fig_style import apply_style

apply_style(font_family="DejaVu Serif", base_font_size=16)

In [2]:
import warnings

import numpy as np
import pandas as pd
import phate
import plotly.graph_objects as go

warnings.filterwarnings("ignore", category=RuntimeWarning)

In [3]:
from gut_former.utils.project_paths import find_data_path, find_figures_path

data_path = find_data_path()
output_path = find_data_path().parent / "output"
figures_path = find_figures_path()
figures_path.mkdir(parents=True, exist_ok=True)

## Read Data & PHATE

In [4]:
latent_df = pd.read_csv(f"{output_path}/latent_sample.csv", index_col=0)

# PHATE embedding (same params as figure3_phate)
phate_op = phate.PHATE(n_components=5, random_state=42, n_pca=10, knn=5)
phate_df = pd.DataFrame(phate_op.fit_transform(latent_df),
                        index=latent_df.index, columns=[f"P{i}" for i in range(5)])

Calculating PHATE...


  Running PHATE on 128 observations and 64 variables.


  Calculating graph and diffusion operator...


    Calculating PCA...


    Calculating KNN search...


...

  Calculated diffusion potential in 0.20 seconds.


  Calculating metric MDS...


    SGD-MDS may not have converged: stress changed by 3.5% in final iterations. Consider increasing n_iter or adjusting learning_rate.


  Calculated metric MDS in 0.14 seconds.


Calculated PHATE in 0.39 seconds.


In [5]:
# Taxonomy -> order level aggregation
Xt = pd.read_csv(f"{data_path}/taxonomy_sample.csv", index_col=0)
order_map = dict(zip(Xt.columns, [c.split("|")[3] for c in Xt.columns]))
family_df = Xt.rename(columns=order_map).T.groupby(level=0).sum().T
family_df.columns = [c.replace("o__", "") for c in family_df.columns]

# Pathways -> MetaCyc top-level category aggregation
Xp = pd.read_csv(f"{data_path}/pathways_sample.csv", index_col=0)
Xp.columns = [c.split(":")[0] for c in Xp.columns]

metacyc = pd.read_csv(f"{data_path}/map_metacyc-pwy_lineage.tsv", sep="\t", header=None)
metacyc.columns = ["pathway", "hierarchy"]
levels = metacyc["hierarchy"].str.split("|", expand=True)
metacyc = pd.concat([metacyc["pathway"], levels], axis=1)
metacyc = metacyc[metacyc["pathway"].isin(Xp.columns)]

Xp_common = Xp[metacyc["pathway"]]
pathway_to_category = dict(zip(metacyc["pathway"], metacyc[0]))
Xp_aggregated = Xp_common.rename(columns=pathway_to_category).T.groupby(level=0).sum().T

## Sankey function

In [6]:
def plot_sankey(exo_df, latent_df, corr_threshold=0.3, width=1200, height=700):
    corr = pd.concat([np.log(exo_df + 1e-3), latent_df], axis=1).corr()
    latent_cols = latent_df.columns.tolist()
    feat_cols = exo_df.columns.tolist()

    links = corr.loc[latent_cols, feat_cols].stack().reset_index()
    links.columns = ["source", "target", "value"]
    links = links[links["value"] > corr_threshold]

    node_labels = latent_cols + feat_cols
    idx = {lab: i for i, lab in enumerate(node_labels)}

    fig = go.Figure(data=[go.Sankey(
        node={"label": node_labels, "pad": 15, "thickness": 20},
        link={"source": links["source"].map(idx), "target": links["target"].map(idx),
              "value": links["value"].abs()},
    )])
    fig.update_layout(height=height, width=width, margin=dict(l=50, r=50, t=50, b=50),
                      font=dict(size=16))
    return fig


def save_sankey(fig, name):
    fig.write_html(f"{figures_path}/{name}.html")
    try:
        fig.write_image(f"{figures_path}/{name}.png", scale=2)
        print(f"saved {name}.png + {name}.html")
    except Exception as e:
        print(f"{name}: PNG export failed ({type(e).__name__}: {e}); HTML saved. "
              f"For PNG run: poetry run plotly_get_chrome")
    fig.show()

# Figure 3D - Sankey: aggregated pathways (MetaCyc)

In [7]:
fig = plot_sankey(Xp_aggregated, phate_df, 0.3)
save_sankey(fig, "Fig3D")

saved Fig3D.png + Fig3D.html


# Figure 3E - Sankey: taxonomy (order level)

In [8]:
fig = plot_sankey(family_df, phate_df, 0.3)
save_sankey(fig, "Fig3E")

saved Fig3E.png + Fig3E.html


# Supplementary Figure 3H - Sankey: all pathways

In [9]:
fig = plot_sankey(Xp, latent_df, 0.5, width=500, height=2500)
save_sankey(fig, "sFig3H")

saved sFig3H.png + sFig3H.html


# Supplementary Figure 3I - Sankey: all species

In [10]:
Xt_species = Xt.copy()
Xt_species.columns = [c.split("|")[-1] for c in Xt_species.columns]
fig = plot_sankey(Xt_species, latent_df, 0.4, width=500, height=2000)
save_sankey(fig, "sFig3I")

saved sFig3I.png + sFig3I.html
